# Backward Propagation

> Based on Stanford CS230 — Lecture Notes 3, Andrew Ng / DeepLearning.AI

Backward propagation (backprop) computes gradients of the loss with respect to all parameters by applying the **chain rule** through the computational graph — from output back to input.

## Learning Objectives

1. Apply the chain rule to derive gradients for each layer
2. Write the vectorised backprop equations for an $L$-layer network
3. Implement gradient checking (numerical vs analytical)
4. Understand the symmetry between forward and backward passes

## The Chain Rule

For a scalar loss $J$ flowing back through $Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}$ and $A^{[l]} = g^{[l]}(Z^{[l]})$:

$$dZ^{[l]} = dA^{[l]} \odot g'^{[l]}\!\left(Z^{[l]}\right)$$

$$dW^{[l]} = \frac{1}{m}\, dZ^{[l]} \left(A^{[l-1]}\right)^T$$

$$db^{[l]} = \frac{1}{m}\, \sum_{\text{cols}} dZ^{[l]}$$

$$dA^{[l-1]} = \left(W^{[l]}\right)^T dZ^{[l]}$$

where $\odot$ is element-wise multiplication and $dX \equiv \frac{\partial J}{\partial X}$.


## Computational Graph

Gradients flow from right to left through the same graph used in the forward pass:

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 680 180" width="680" height="180" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <!-- Forward pass arrow (top) -->
  <defs>
    <marker id="arr-fwd" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto">
      <path d="M0,0 L0,6 L8,3 z" fill="#5b9bd5"/>
    </marker>
    <marker id="arr-bwd" markerWidth="8" markerHeight="8" refX="2" refY="3" orient="auto">
      <path d="M8,0 L8,6 L0,3 z" fill="#e05c5c"/>
    </marker>
  </defs>
  <text x="340" y="18" text-anchor="middle" fill="#5b9bd5" font-size="11" font-weight="bold">Forward pass →</text>
  <text x="340" y="168" text-anchor="middle" fill="#e05c5c" font-size="11" font-weight="bold">← Backward pass (gradients)</text>
  <!-- Nodes: X, Z[1], A[1], Z[2], A[2], J -->
  <!-- X -->
  <rect x="20"  y="65" width="60" height="50" rx="6" fill="#5b9bd5" fill-opacity="0.12" stroke="#5b9bd5" stroke-width="1.8"/>
  <text x="50"  y="88" text-anchor="middle" fill="#3a7bbf" font-size="12" font-weight="bold">A&#x2070;</text>
  <text x="50"  y="104" text-anchor="middle" fill="#3a7bbf" font-size="10">= X</text>
  <!-- Z[1] -->
  <rect x="140" y="65" width="70" height="50" rx="6" fill="#7ecba1" fill-opacity="0.12" stroke="#7ecba1" stroke-width="1.8"/>
  <text x="175" y="88"  text-anchor="middle" fill="#3d7a5e" font-size="11" font-weight="bold">Z&#x00B9;</text>
  <text x="175" y="104" text-anchor="middle" fill="#3d7a5e" font-size="10">W&#x00B9;A&#x2070;+b&#x00B9;</text>
  <!-- A[1] -->
  <rect x="270" y="65" width="70" height="50" rx="6" fill="#7ecba1" fill-opacity="0.12" stroke="#7ecba1" stroke-width="1.8"/>
  <text x="305" y="88"  text-anchor="middle" fill="#3d7a5e" font-size="11" font-weight="bold">A&#x00B9;</text>
  <text x="305" y="104" text-anchor="middle" fill="#3d7a5e" font-size="10">g&#x00B9;(Z&#x00B9;)</text>
  <!-- Z[2] -->
  <rect x="400" y="65" width="70" height="50" rx="6" fill="#7ecba1" fill-opacity="0.12" stroke="#7ecba1" stroke-width="1.8"/>
  <text x="435" y="88"  text-anchor="middle" fill="#3d7a5e" font-size="11" font-weight="bold">Z&#x00B2;</text>
  <text x="435" y="104" text-anchor="middle" fill="#3d7a5e" font-size="10">W&#x00B2;A&#x00B9;+b&#x00B2;</text>
  <!-- A[2] / ŷ -->
  <rect x="530" y="65" width="60" height="50" rx="6" fill="#e05c5c" fill-opacity="0.12" stroke="#e05c5c" stroke-width="1.8"/>
  <text x="560" y="88"  text-anchor="middle" fill="#b03a3a" font-size="11" font-weight="bold">&#x177;</text>
  <text x="560" y="104" text-anchor="middle" fill="#b03a3a" font-size="10">&#x03C3;(Z&#x00B2;)</text>
  <!-- Loss -->
  <rect x="620" y="72" width="45" height="36" rx="6" fill="#f4b942" fill-opacity="0.15" stroke="#f4b942" stroke-width="1.8"/>
  <text x="642" y="94" text-anchor="middle" fill="#a07010" font-size="11" font-weight="bold">J</text>
  <!-- Forward arrows -->
  <line x1="80"  y1="90" x2="137" y2="90" stroke="#5b9bd5" stroke-width="1.5" marker-end="url(#arr-fwd)"/>
  <line x1="210" y1="90" x2="267" y2="90" stroke="#5b9bd5" stroke-width="1.5" marker-end="url(#arr-fwd)"/>
  <line x1="340" y1="90" x2="397" y2="90" stroke="#5b9bd5" stroke-width="1.5" marker-end="url(#arr-fwd)"/>
  <line x1="470" y1="90" x2="527" y2="90" stroke="#5b9bd5" stroke-width="1.5" marker-end="url(#arr-fwd)"/>
  <line x1="590" y1="90" x2="617" y2="90" stroke="#5b9bd5" stroke-width="1.5" marker-end="url(#arr-fwd)"/>
  <!-- Backward arrows (below) -->
  <line x1="137" y1="115" x2="80"  y2="115" stroke="#e05c5c" stroke-width="1.5" stroke-dasharray="5,2" marker-end="url(#arr-bwd)"/>
  <line x1="267" y1="115" x2="210" y2="115" stroke="#e05c5c" stroke-width="1.5" stroke-dasharray="5,2" marker-end="url(#arr-bwd)"/>
  <line x1="397" y1="115" x2="340" y2="115" stroke="#e05c5c" stroke-width="1.5" stroke-dasharray="5,2" marker-end="url(#arr-bwd)"/>
  <line x1="527" y1="115" x2="470" y2="115" stroke="#e05c5c" stroke-width="1.5" stroke-dasharray="5,2" marker-end="url(#arr-bwd)"/>
  <line x1="617" y1="115" x2="590" y2="115" stroke="#e05c5c" stroke-width="1.5" stroke-dasharray="5,2" marker-end="url(#arr-bwd)"/>
  <!-- gradient labels -->
  <text x="108"  y="128" text-anchor="middle" fill="#e05c5c" font-size="9">dA&#x2070;</text>
  <text x="238"  y="128" text-anchor="middle" fill="#e05c5c" font-size="9">dA&#x00B9;</text>
  <text x="368"  y="128" text-anchor="middle" fill="#e05c5c" font-size="9">dZ&#x00B9;</text>
  <text x="498"  y="128" text-anchor="middle" fill="#e05c5c" font-size="9">dZ&#x00B2;</text>
  <text x="603"  y="128" text-anchor="middle" fill="#e05c5c" font-size="9">dJ/d&#x177;</text>
</svg>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

# -------------------------------------------------------------------
# 2-layer network: forward + backward from scratch + gradient check
# -------------------------------------------------------------------
def sigmoid(z):      return 1 / (1 + np.exp(-z))
def sigmoid_d(z):    s = sigmoid(z); return s * (1 - s)
def relu(z):         return np.maximum(0, z)
def relu_d(z):       return (z > 0).astype(float)

def init_params(dims, seed=42):
    np.random.seed(seed)
    p = {}
    for l in range(1, len(dims)):
        p[f'W{l}'] = np.random.randn(dims[l], dims[l-1]) * np.sqrt(2 / dims[l-1])
        p[f'b{l}'] = np.zeros((dims[l], 1))
    return p

def forward(X, params):
    W1, b1 = params['W1'], params['b1']
    W2, b2 = params['W2'], params['b2']
    Z1 = W1 @ X + b1
    A1 = relu(Z1)
    Z2 = W2 @ A1 + b2
    A2 = sigmoid(Z2)
    cache = (X, Z1, A1, Z2, A2)
    return A2, cache

def compute_cost(A2, Y):
    m = Y.shape[1]
    return -np.mean(Y * np.log(A2 + 1e-8) + (1 - Y) * np.log(1 - A2 + 1e-8))

def backward(cache, params, Y):
    X, Z1, A1, Z2, A2 = cache
    m = Y.shape[1]
    W2 = params['W2']

    dA2 = -(Y / (A2 + 1e-8) - (1 - Y) / (1 - A2 + 1e-8))
    dZ2 = dA2 * sigmoid_d(Z2)
    dW2 = (dZ2 @ A1.T) / m
    db2 = np.mean(dZ2, axis=1, keepdims=True)

    dA1 = W2.T @ dZ2
    dZ1 = dA1 * relu_d(Z1)
    dW1 = (dZ1 @ X.T) / m
    db1 = np.mean(dZ1, axis=1, keepdims=True)

    return {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}

# --- Train on toy data ---
np.random.seed(0)
m = 300
X = np.vstack([np.random.randn(2, m//2) + [[2], [2]],
               np.random.randn(2, m//2) + [[-2], [-2]]]).T
X = X.T
Y = np.array([1]*(m//2) + [0]*(m//2)).reshape(1, -1)

params = init_params([2, 8, 1])
lr, costs = 0.5, []
for i in range(2000):
    A2, cache = forward(X, params)
    costs.append(compute_cost(A2, Y))
    grads = backward(cache, params, Y)
    for k in params:
        params[k] -= lr * grads[f'd{k}']

# --- Gradient checking ---
def params_to_vec(p):
    return np.concatenate([v.flatten() for v in p.values()])

def vec_to_params(vec, dims):
    p, idx = {}, 0
    for l in range(1, len(dims)):
        n_W = dims[l] * dims[l-1]
        n_b = dims[l]
        p[f'W{l}'] = vec[idx:idx+n_W].reshape(dims[l], dims[l-1]); idx += n_W
        p[f'b{l}'] = vec[idx:idx+n_b].reshape(dims[l], 1);           idx += n_b
    return p

eps = 1e-5
dims = [2, 8, 1]
p_test = init_params(dims, seed=7)
A2_t, cache_t = forward(X, p_test)
grads_t = backward(cache_t, p_test, Y)
grad_vec = params_to_vec({f'd{k}': v for k, v in grads_t.items()})

theta = params_to_vec(p_test)
grad_num = np.zeros_like(theta)
for i in range(len(theta)):
    tp, tm = theta.copy(), theta.copy()
    tp[i] += eps; tm[i] -= eps
    Jp = compute_cost(forward(X, vec_to_params(tp, dims))[0], Y)
    Jm = compute_cost(forward(X, vec_to_params(tm, dims))[0], Y)
    grad_num[i] = (Jp - Jm) / (2 * eps)

diff = np.linalg.norm(grad_num - grad_vec) / (np.linalg.norm(grad_num) + np.linalg.norm(grad_vec))
print(f"Gradient check relative difference: {diff:.2e}  {'✓ PASS' if diff < 1e-5 else '✗ FAIL'}")

# --- Plot training cost + decision boundary ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Backward Propagation — Training a 2-Layer Network', fontsize=13, fontweight='bold')

axes[0].plot(costs, color=P[0], lw=2)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Cost J')
axes[0].set_title(f'Training Curve  (final J = {costs[-1]:.4f})')
axes[0].grid(True)

ax = axes[1]
h = 0.05
xx, yy = np.meshgrid(np.arange(-5, 5, h), np.arange(-5, 5, h))
grid = np.c_[xx.ravel(), yy.ravel()].T
probs, _ = forward(grid, params)
probs = probs.reshape(xx.shape)
ax.contourf(xx, yy, probs, levels=50, cmap='RdBu_r', alpha=0.5)
ax.scatter(X[0, Y[0]==1], X[1, Y[0]==1], color=P[0], s=20, alpha=0.7, label='y=1')
ax.scatter(X[0, Y[0]==0], X[1, Y[0]==0], color=P[1], s=20, alpha=0.7, label='y=0')
ax.set_title('Learned Decision Boundary')
ax.legend(fontsize=9)
ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
ax.grid(True)

plt.tight_layout()
plt.show()

preds = (forward(X, params)[0] > 0.5).astype(int)
print(f"Training accuracy: {np.mean(preds == Y)*100:.1f}%")


## Gradient Checking

Numerical gradient checking verifies your analytical gradients:

$$\frac{\partial J}{\partial \theta_i} \approx \frac{J(\theta + \varepsilon \mathbf{e}_i) - J(\theta - \varepsilon \mathbf{e}_i)}{2\varepsilon}$$

Compare with $\|g_{num} - g_{analytic}\|_2 \,/\, \bigl(\|g_{num}\|_2 + \|g_{analytic}\|_2\bigr)$. If this ratio is $< 10^{-5}$, the gradient is correct.

> **Never use gradient checking during training** — it is $O(n\_params \times m)$ and too slow. Run it once during debugging on a small batch.
